# 📘 **Panorama General: APIs en el Web Scraping**

---

## 1. 🧩 **¿Qué es una API?**

Una **API (Application Programming Interface)** es un conjunto de reglas que permite que dos sistemas intercambien datos de manera estructurada, estandarizada y controlada.

### En el contexto de scraping

- Una API permite obtener información **sin necesidad de extraer HTML**.
- Proporciona datos ya estructurados (JSON, XML), evitando parseo manual.
- Es la forma **más limpia, eficiente y estable** de obtener datos de un servicio web.

### El modelo básico

```pgsql
CLIENTE (tu script) → solicita datos → API SERVER → devuelve JSON
```

---

## 2. 🧩 **¿Qué es “API Scraping”?**

**API Scraping** se refiere al uso de puntos finales (*endpoints*) públicos o semipúblicos para extraer datos que una web ofrece internamente, aun cuando no haya una documentación oficial.

Incluye dos escenarios:

### ✔ **1. Uso de APIs oficiales (documentadas)**

Como:

- Twitter/X API
- Reddit API
- Spotify API
- GitHub API
- HuggingFace API

Ventajas: legales, estables, con límites claros.

### ✔ **2. Uso de APIs internas (no documentadas)**

- Las webs modernas cargan datos vía peticiones XHR/JSON en segundo plano.
- Puedes ver esas llamadas en “Network → Fetch/XHR” del navegador.
- Puedes reconstruir esas llamadas desde Python.

Esto **no es ilegal**, pero está sujeto a:

- Términos de uso
- Límites de acceso
- Políticas de automatización

---

## 3. 🔌 **¿Cómo funciona técnicamente una API?**

Cuando haces una llamada a una API:

1. Se abre una conexión TCP/IP al servidor.
2. Se envía una solicitud HTTP o HTTPS:

   - Verbo (GET, POST, PUT, DELETE…)
   - Headers (User-Agent, Authorization, Accept, etc.)
   - Parámetros (query string o body)
3. El servidor procesa y responde:

   - Código de estado (200, 403, 404…)
   - Datos estructurados (JSON)

### Ejemplo real de comunicación

```
GET /v1/users?limit=20 HTTP/1.1
Host: api.ejemplo.com
Authorization: Bearer <token>
Accept: application/json
```

Respuesta:

```json
{
  "data": [
    {"id": 1, "name": "Ana"},
    {"id": 2, "name": "Carlos"}
  ],
  "meta": {
    "count": 2,
    "page": 1
  }
}
```

### Componentes esenciales

#### ✔ **Cliente**

Tu script en Python (por ejemplo, usando `requests`).

#### ✔ **Servidor**

El servicio que ofrece los datos. Tiene reglas claras: quién puede pedir datos, cuántas veces y en qué formato.

#### ✔ **Endpoints**

Son las “direcciones” donde solicitas información:

```
/users
/users/123
/products
/products/45/reviews
```

### ✔ **Métodos HTTP**

- **GET** → obtener datos
- **POST** → enviar datos (formularios, autenticación)
- **PUT/PATCH** → actualizar
- **DELETE** → eliminar

### ✔ **Representación de datos**

Normalmente **JSON**
Ocasionalmente:

- XML
- CSV
- BINARIO (PDF, imágenes, etc.)

### ✔ **Códigos de estado**

- **200** OK
- **404** No existe
- **401** No autorizado
- **429** Demasiadas peticiones

---

## 4. 🧱 **Elementos clave de una API moderna**

### ✔ Endpoints

Rutas específicas donde se solicitan datos:

```
/v1/users
/v1/users/{id}
/v1/products
```

### ✔ Métodos HTTP

- **GET:** obtener datos
- **POST:** enviar datos
- **PUT/PATCH:** actualizar
- **DELETE:** eliminar

### 🛑 **Errores HTTP y cómo manejarlos**

#### ✔ **Errores comunes**

- **400** → petición mal formada
- **401** → falta autenticación
- **403** → prohibido
- **404** → no existe
- **500** → error del servidor
- **503** → servicio saturado

#### Manejo básico

```python
if resp.status_code == 200:
    data = resp.json()
else:
    print("Error:", resp.status_code)
```

#### Manejo robusto

- Reintentos
- Backoff
- Registro de errores
- Guardar tokens renovados

### ✔ Headers

Usados para identificar al cliente y controlar el formato:

- `Authorization`
- `User-Agent`
- `Accept: application/json`

### ✔ Parámetros

- **Query params:** `?page=1&limit=50`
- **Body params** (POST/PUT): JSON que envías al servidor.

### ⏳ **Rate limits, backoff y manejo de sesiones**

Para evitar bloqueo:

#### ✔ **Rate Limits**

La API puede permitir:

- 60 requests/minuto
- 1000 por día
- etc.

Cuando se exceden → error **429 Too Many Requests**.

#### ✔ **Backoff (espera progresiva)**

Si te bloquean, espera escalonadamente:

```python
import time

wait = 1
for intento in range(5):
    resp = requests.get(url)
    if resp.status_code == 429:
        time.sleep(wait)
        wait *= 2
    else:
        break
```

#### ✔ **Uso de sesiones**

Una sesión mantiene conexión y cookies, acelerando la comunicación:

```python
s = requests.Session()
resp = s.get(url)
```

### 🔐 **Autenticación en APIs**

#### ✔ **API Key**

Una cadena secreta que identifica a tu aplicación.

```python
headers = {"Authorization": "Api-Key TU_KEY"}
```

#### ✔ **Bearer Token**

Token que representa a un usuario autenticado.

```python
headers = {"Authorization": "Bearer TOKEN"}
```

#### ✔ **OAuth2**

Usado en servicios grandes (Google, GitHub, Discord).
Implica redirecciones, permisos y tokens temporales.

#### ✔ **Cookies / Sesión**

En APIs internas puede que la web genere una cookie, y debes enviarla para que la API te reconozca:

```python
session = requests.Session()
session.cookies.update({"sessionid": "..."} )
```

#### Firmas

- AWS
- Binance
- MercadoLibre

---

## 5. 🔍 **API vs Scraping tradicional (HTML)**

| Criterio        | API                             | HTML Scraping                        |
| --------------- | ------------------------------- | ------------------------------------ |
| Estructura      | Datos limpios                   | Depende del DOM                      |
| Estabilidad     | Alta                            | Baja (se rompe con cambios visuales) |
| Eficiencia      | Muy rápida                      | Más lenta                            |
| Requiere parseo | No                              | Sí (BeautifulSoup)                   |
| Limitaciones    | Fuertes límites y autenticación | Detectables por anti-bots            |
| Legalidad       | Generalmente más clara          | Ambiguo si se ignoran políticas      |

---

## 6. 🧠 **Ventajas del uso de APIs**

- **Velocidad:** procesan datos directamente, sin HTML.
- **Fiabilidad:** no dependen de cambios en la interfaz visual.
- **Menos complejidad:** no necesitas selenium, renderizado ni DOM.
- **Mayor volumen de datos:** muchas APIs permiten descargar masivamente.
- **Legalmente más seguras:** cuando son oficiales.

---

## 7. ⚠️ **Desventajas**

- Requieren autenticación o registro.
- Tienen límites por tiempo.
- Algunas APIs son de pago.
- Los endpoints pueden cambiar sin aviso.
- Hay APIs que ocultan datos que sí están presentes en el sitio web.

---

## 8. 🛠️ **Cómo identificar si un sitio usa API (incluso sin documentarla)**

Pasos:

1. Abre el navegador → **F12** → *DevTools*
2. Ve a **Network**.
3. Filtra por:

   - **Fetch/XHR**
   - **JSON**
4. Observa las solicitudes:

   - URL
   - Query params (`?page=1`)
   - Headers
   - Body
   - Respuesta JSON

5. Ejemplo:

Si ves algo como:

```nginx
GET https://example.com/api/products?page=1
```

Y la respuesta es JSON:
👉 ya descubriste un endpoint utilizable desde Python.

---

## 9. 📚 **Herramientas comunes para consumir APIs desde Python**

### ✔ **requests** (el rey del scraping API)

- Para APIs REST
- Fácil y rápido
- Control total sobre headers, cookies, sesiones

### ✔ **httpx**

- Asíncrono y moderno

### ✔ **aiohttp**

- Excelente para grandes volúmenes asincrónicos

### ✔ SDKs oficiales

- Google Cloud SDK
- AWS boto3
- Telegram API libraries
- GitHub PyGithub

---

## 10. ↔️ **Paginación y acceso a grandes volúmenes**

Las APIs no entregan miles de registros de una vez. Usan paginación para evitar saturarse.

Tres formas comunes:

### ✔ **Paginación por página**

```
?page=1
?page=2
```

### ✔ **Paginación por offset**

```
?offset=0&limit=50
?offset=50&limit=50
```

### ✔ **Cursor-based**

La más moderna, entregan un “cursor” a la siguiente página:

```json
{"next_cursor": "XYZ123"}
```

### Ejemplo

```python
page = 1
while True:
    r = requests.get(url, params={"page": page})
    data = r.json()

    procesar(data)

    if not data["has_next"]:
        break
    page += 1
```

---

## 11. 🧰 **Ejemplo general de acceso a una API desde Python**

```python
import requests

url = "https://api.ejemplo.com/v1/users"
headers = {
    "Authorization": "Bearer TU_TOKEN",
    "Accept": "application/json"
}

params = {
    "limit": 50,
    "page": 1
}

response = requests.get(url, headers=headers, params=params)

data = response.json()

print(data)
```

---

## 12. 🔐 **Aspectos éticos y legales**

### ✔ APIs oficiales

- Suelen permitir uso de datos siempre que respetes:

  - Rate limits
  - No redistribuir datos sensibles
  - No automatizar acciones prohibidas

### ✔ APIs internas no documentadas

- Debes verificar:

  - Términos del servicio
  - Política de automatización
  - Restricciones de uso comercial

### Buenas prácticas

- Limitar la velocidad de tus requests
- No exponer API keys
- Respetar robots.txt si también scrapeas HTML
- No recolectar información personal sin anonimizar

# **1. Cómo funciona la API de Telegram (visión global)**

Telegram ofrece **dos APIs** principales:

## **1. Telegram Bot API**

- Es la API más usada para proyectos de scraping, automatización y bots.
- Permite crear bots que pueden **leer mensajes**, **enviar contenido**, **extraer información de grupos/canales**, etc.
- Es **más fácil**, pero tiene **limitaciones**:

  - Solo accedes a mensajes donde el bot está presente.
  - No puedes leer mensajes de usuarios privados.
  - En grupos/canales, solo puedes leer lo que llega *después* de que el bot fue agregado (no histórico).

## **2. Telegram Client API (TDLib / Telegram Core API)**

- Es la API que usan clientes de Telegram “completos”.
- Permite acceder a **tu propia cuenta personal** como si fueras un cliente real.
- Ofrece más funciones que el Bot API:

  - Leer mensajes ya existentes en grupos/canales.
  - Acceder a tu lista de chats y contactos.
  - Descargar multimedia historial.
  - Es **mucho más compleja**, basada en MTProto.
  - Requiere mayor cuidado legal y ético.

---

# **2. Autenticación en la API de Telegram**

## **2.1 Autenticación en Bot API**

Es el método más simple.

### **Pasos:**

1. Abre Telegram y busca **@BotFather**
2. Ejecuta `/newbot`
3. Asigna nombre y @username
4. Recibirás un **API token** con este formato:

```
1234567890:AAEcc-ds87asd9sd8f0000fsdfsdf
```

Ese token te sirve para hacer requests a la API, ejemplo:

```
https://api.telegram.org/bot<TOKEN>/getUpdates
```

### **Nivel de dificultad:** muy bajo

### **Seguridad:** guardar el token en variables de entorno.

---

## **2.2 Autenticación en Client API (MTProto / TDLib)**

Si deseas acceder al contenido como un usuario real (por ejemplo, scraping de canales públicos):

### **Pasos:**

1. Ir a [https://my.telegram.org](https://my.telegram.org)
2. Iniciar sesión con tu número telefónico.
3. Crear un **API ID** y **API HASH**.

Datos típicos:

```
api_id = 1234567
api_hash = "ae8a879a98bdfe98...."
```

4. Usar una librería como **Telethon**, **Pyrogram**, o **TDLib** para autenticarse.

Ejemplo Telethon (solo demostrativo):

```python
from telethon import TelegramClient

client = TelegramClient('session_name', api_id, api_hash)
await client.start()
```

El primer inicio pedirá un **código por SMS**, como en el login normal.

### **Ventajas**

- Puedes leer historial.
- Puedes scrapear canales públicos completos.

### **Desventaja**

- Riesgo de bloqueo si Telegram detecta actividad sospechosa.
- Reglas de uso más estrictas.

---

# **3. Reglas básicas al trabajar con la API de Telegram**

Telegram tiene políticas estrictas para evitar abuso:

## **3.1 Límites de rate-limit**

- La Bot API permite ~30 solicitudes por segundo.
- La Client API no da un número exacto, pero si haces demasiadas:

  - Obtienes *FloodWaitError*
  - Tu cuenta puede quedar temporalmente bloqueada.

**Solución:** usar pausas (`sleep(1)`), y paginar datos.

---

## **3.2 Reglas sobre obtención de datos**

Telegram controla quién puede acceder a qué:

### **Bot API**

- Solo recibes mensajes donde el bot está presente.
- No puedes obtener historial anterior.
- No puedes leer mensajes privados sin consentimiento.

### **Client API**

- Puedes scrapear canales públicos.
- Puedes acceder a mensajes retroactivos.
- Pero **no puedes descargar ni compartir datos personales sensibles**.

---

## **3.3 Prohibiciones expresas**

Según los Términos de Uso:

- No puedes usar Telegram para construir bases de datos de reconocimiento facial.
- No puedes vender o redistribuir datos de usuarios sin consentimiento.
- No puedes scrapear contenido privado.
- No puedes automatizar acciones masivas (spam, joins repetidos).

---

# **4. Consejos prácticos**

1. **Para proyectos universitarios** usa *Telegram Bot API* si es suficiente.
2. Si necesitas **historial** o scraping avanzado, usa *Telethon*, pero cuidadosamente.
3. Respeta las **políticas de privacidad**.
4. No hagas scraping masivo sin pausa.
5. Documenta siempre:

   - fecha de acceso
   - canal
   - tipo de información
   - nivel de anonimización